# Objectives

### English

- Build current match vectors for any pair of qualified national teams.
- Retrieve the corresponding Current Team Vectors.
- Create home and away feature representations.
- Generate difference, relative difference, and sum features.
- Align the resulting feature set with the historical training dataset.
- Perform integrity checks to ensure feature consistency.
- Export prediction-ready current match vectors.

### Español

- Construir vectores de partidos actuales para cualquier par de selecciones clasificadas.
- Recuperar los Current Team Vectors correspondientes.
- Crear las representaciones de estadísticas para el equipo local y visitante.
- Generar variables de diferencia, diferencia relativa y suma.
- Alinear el conjunto de variables con el dataset histórico utilizado para entrenar el modelo.
- Realizar verificaciones de integridad para garantizar la consistencia de las features.
- Exportar Current Match Vectors listos para predicción.

# Configuration

## Imports

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

## Paths

In [2]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

CURRENT_TEAM_VECTOR_PATH = (
    PROCESSED_DIR / "current_team_vectors_v01.csv"
)

HISTORICAL_MATCH_VECTOR_PATH = (
    PROCESSED_DIR / "match_vector_v03.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR / "current_match_vector_v01.csv"
)

## Utility Functions

In [3]:
def get_team_vector(
    team_name,
    team_vectors,
    statistical_columns,
    team_id_column="country_clean",
):
    """
    Retrieve the statistical vector of a national team.

    Parameters
    ----------
    team_name : str
        Name of the national team.

    team_vectors : pandas.DataFrame
        Dataset containing one aggregated vector per team.

    statistical_columns : list of str
        Statistical feature columns to retrieve.

    team_id_column : str, default="country_clean"
        Column used to identify each team.

    Returns
    -------
    pandas.Series
        Statistical vector of the requested team.

    Raises
    ------
    ValueError
        If the team is not available or appears more than once.
    """

    team_rows = team_vectors.loc[
        team_vectors[team_id_column].eq(team_name)
    ]

    if team_rows.empty:
        available_teams = sorted(
            team_vectors[team_id_column].unique()
        )

        raise ValueError(
            f"Team '{team_name}' was not found. "
            f"Available teams: {available_teams}"
        )

    if len(team_rows) != 1:
        raise ValueError(
            f"Expected one vector for '{team_name}', "
            f"but found {len(team_rows)}."
        )

    return (
        team_rows
        .iloc[0]
        .loc[statistical_columns]
        .copy()
    )

In [4]:
def build_current_match_vector(
    home_team,
    away_team,
    team_vectors,
    statistical_columns,
    team_id_column="country_clean",
):
    """
    Build a base current match vector from two national teams.

    Parameters
    ----------
    home_team : str
        Name of the home national team.

    away_team : str
        Name of the away national team.

    team_vectors : pandas.DataFrame
        Dataset containing one aggregated vector per team.

    statistical_columns : list of str
        Statistical feature columns used to represent each team.

    team_id_column : str, default="country_clean"
        Column used to identify each team.

    Returns
    -------
    pandas.DataFrame
        Single-row match vector containing home and away features.

    Raises
    ------
    ValueError
        If the same team is used as both home and away.
    """

    if home_team == away_team:
        raise ValueError(
            "Home team and away team must be different."
        )

    home_vector = get_team_vector(
        team_name=home_team,
        team_vectors=team_vectors,
        statistical_columns=statistical_columns,
        team_id_column=team_id_column,
    ).astype(float)

    away_vector = get_team_vector(
        team_name=away_team,
        team_vectors=team_vectors,
        statistical_columns=statistical_columns,
        team_id_column=team_id_column,
    ).astype(float)

    home_vector.index = [
        f"home_{column}"
        for column in home_vector.index
    ]

    away_vector.index = [
        f"away_{column}"
        for column in away_vector.index
    ]

    match_vector = pd.concat(
        [
            pd.Series(
                {
                    "home_team": home_team,
                    "away_team": away_team,
                }
            ),
            home_vector,
            away_vector,
        ]
    )

    return match_vector.to_frame().T

In [21]:
def get_team_statistics(df):
    """
    Return the numeric statistics available for both
    home and away teams.

    Identifiers, team names, target variables, and
    match results are excluded.
    """

    excluded_stats = {
        "match_id",
        "team_name",
        "team",
        "score",
        "year",
    }

    home_stats = {
        column.replace("home_", "", 1)
        for column in df.columns
        if column.startswith("home_")
    }

    away_stats = {
        column.replace("away_", "", 1)
        for column in df.columns
        if column.startswith("away_")
    }

    common_stats = sorted(
        home_stats.intersection(away_stats)
    )

    valid_stats = [
        stat
        for stat in common_stats
        if stat not in excluded_stats
        and pd.api.types.is_numeric_dtype(
            df[f"home_{stat}"]
        )
        and pd.api.types.is_numeric_dtype(
            df[f"away_{stat}"]
        )
    ]

    return valid_stats

In [22]:
def create_difference_features(df):
    """
    Create absolute differences between home and away
    team statistics.

    Formula
    -------
    diff_X = home_X - away_X
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        df_c[f"diff_{stat}"] = (
            df_c[f"home_{stat}"]
            - df_c[f"away_{stat}"]
        )

    return df_c

In [23]:
def create_relative_difference_features(
    df,
    epsilon=1e-6,
):
    """
    Create relative differences between home and away
    team statistics.

    Formula
    -------
    relative_diff_X =
        (home_X - away_X)
        / (home_X + away_X + epsilon)
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        home_col = f"home_{stat}"
        away_col = f"away_{stat}"
        new_col = f"relative_diff_{stat}"

        df_c[new_col] = (
            df_c[home_col] - df_c[away_col]
        ) / (
            df_c[home_col]
            + df_c[away_col]
            + epsilon
        )

    return df_c

In [24]:
def create_sum_features(df):
    """
    Create combined sums of home and away
    team statistics.

    Formula
    -------
    sum_X = home_X + away_X
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        home_col = f"home_{stat}"
        away_col = f"away_{stat}"
        new_col = f"sum_{stat}"

        df_c[new_col] = (
            df_c[home_col] + df_c[away_col]
        )

    return df_c

### Master Function

In [70]:
def build_prediction_ready_match_vector(
    home_team,
    away_team,
    team_vectors,
    statistical_columns,
    historical_feature_columns,
):
    base_vector = build_current_match_vector(
        home_team=home_team,
        away_team=away_team,
        team_vectors=team_vectors,
        statistical_columns=statistical_columns,
    )

    home_away_columns = [
        column
        for column in base_vector.columns
        if column.startswith(("home_", "away_"))
        and column not in {"home_team", "away_team"}
    ]

    base_vector[home_away_columns] = (
        base_vector[home_away_columns]
        .astype(float)
    )

    engineered_vector = (
        base_vector
        .pipe(create_difference_features)
        .pipe(create_relative_difference_features)
        .pipe(create_sum_features)
    )

    missing_features = [
        column
        for column in historical_feature_columns
        if column not in engineered_vector.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing required historical features: "
            f"{missing_features}"
        )

    return engineered_vector[
        ["home_team", "away_team"]
        + historical_feature_columns
    ].copy()

## Load Data

In [5]:
current_team_vectors = pd.read_csv(
    CURRENT_TEAM_VECTOR_PATH
)

historical_match_vectors = pd.read_csv(
    HISTORICAL_MATCH_VECTOR_PATH
)

### Initial Overview

In [6]:
print("Current Team Vectors:")
display(current_team_vectors.head())

print()

print("Historical Match Vectors:")
display(historical_match_vectors.head())

Current Team Vectors:


,country_clean,squad_size,matched_players,unmatched_players,coverage,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,...,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
0,Algeria,26,13,13,0.500000,9.0,1.0,1753.0,135.0,56.0,...,3.0,3.0,471.0,2.0,33.0,2.0,2.0,1.0,0.0,0.0
1,Argentina,26,20,6,0.769231,91.0,28.0,26859.0,1799.0,591.0,...,18.0,18.0,4893.0,19.0,1022.0,11.0,5.0,2.0,0.0,0.0
2,Australia,26,8,18,0.307692,8.0,1.0,1538.0,178.0,86.0,...,1.0,1.0,538.0,2.0,23.0,0.0,0.0,1.0,0.0,0.0
3,Austria,26,20,6,0.769231,43.0,4.0,4846.0,508.0,222.0,...,3.0,3.0,1916.0,6.0,139.0,3.0,1.0,0.0,0.0,0.0
4,Belgium,26,15,11,0.576923,23.0,1.0,5399.0,509.0,159.0,...,2.0,2.0,1477.0,5.0,142.0,4.0,1.0,0.0,0.0,0.0



Historical Match Vectors:


,match_id,match_date,kick_off,year,home_team,away_team,home_score,away_score,competition_stage,home_Ball Receipt*,...,sum_Interception,sum_Miscontrol,sum_Offside,sum_Own Goal Against,sum_Pass,sum_Player Off,sum_Player On,sum_Pressure,sum_Shield,sum_Shot
0,7525,2018-06-14,15:00:00.000,2018,Russia,Saudi Arabia,5,0,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,7578,2018-06-15,12:00:00.000,2018,Egypt,Uruguay,0,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7577,2018-06-15,15:00:00.000,2018,Morocco,Iran,0,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7576,2018-06-15,18:00:00.000,2018,Portugal,Spain,3,3,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,7530,2018-06-16,10:00:00.000,2018,France,Australia,2,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
print("Current Team Vectors:", current_team_vectors.shape)
print("Historical Match Vectors:", historical_match_vectors.shape)

Current Team Vectors: (48, 32)
Historical Match Vectors: (128, 134)


In [8]:
assert CURRENT_TEAM_VECTOR_PATH.exists()
assert HISTORICAL_MATCH_VECTOR_PATH.exists()

print("Datasets loaded successfully.")

Datasets loaded successfully.


# Inspect Team Vector Structure

Before constructing match-level representations, the structure of the Current Team Vector dataset is inspected.

The dataset contains:

- Team identification.
- Roster coverage metadata.
- Aggregated historical statistical features.

Only the statistical columns will be used to construct the home and away team representations.

### Español

Antes de construir las representaciones a nivel partido, se inspecciona la estructura del dataset Current Team Vector.

El dataset contiene:

- Identificación de la selección.
- Metadatos de cobertura del plantel.
- Variables estadísticas históricas agregadas.

Solamente las columnas estadísticas se utilizarán para construir las representaciones del equipo local y visitante.

## Column Inspection

In [9]:
current_team_vectors.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country_clean      48 non-null     str    
 1   squad_size         48 non-null     int64  
 2   matched_players    48 non-null     int64  
 3   unmatched_players  48 non-null     int64  
 4   coverage           48 non-null     float64
 5   50/50              48 non-null     float64
 6   Bad Behaviour      48 non-null     float64
 7   Ball Receipt*      48 non-null     float64
 8   Ball Recovery      48 non-null     float64
 9   Block              48 non-null     float64
 10  Carry              48 non-null     float64
 11  Clearance          48 non-null     float64
 12  Dispossessed       48 non-null     float64
 13  Dribble            48 non-null     float64
 14  Dribbled Past      48 non-null     float64
 15  Duel               48 non-null     float64
 16  Foul Committed     48 non-null     floa

In [10]:
current_team_vectors.columns.tolist()

['country_clean',
 'squad_size',
 'matched_players',
 'unmatched_players',
 'coverage',
 '50/50',
 'Bad Behaviour',
 'Ball Receipt*',
 'Ball Recovery',
 'Block',
 'Carry',
 'Clearance',
 'Dispossessed',
 'Dribble',
 'Dribbled Past',
 'Duel',
 'Foul Committed',
 'Foul Won',
 'Goal Keeper',
 'Interception',
 'Miscontrol',
 'Pass',
 'Player Off',
 'Player On',
 'Pressure',
 'Shield',
 'Shot',
 'Error',
 'Offside',
 'Own Goal Against',
 'Camera On',
 'Own Goal For']

## Define Metadata Columns

In [11]:
TEAM_ID_COLUMN = "country_clean"

COVERAGE_COLUMNS = [
    "squad_size",
    "matched_players",
    "unmatched_players",
    "coverage",
]

TEAM_METADATA_COLUMNS = [
    TEAM_ID_COLUMN,
    *COVERAGE_COLUMNS,
]

## Identify statistical columns

In [12]:
statistical_columns = [
    column
    for column in current_team_vectors.columns
    if column not in TEAM_METADATA_COLUMNS
]

print("Statistical columns:", len(statistical_columns))
print()
print(statistical_columns)

Statistical columns: 27

['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Error', 'Offside', 'Own Goal Against', 'Camera On', 'Own Goal For']


## Basic Structure Checks

In [13]:
assert TEAM_ID_COLUMN in current_team_vectors.columns

assert all(
    column in current_team_vectors.columns
    for column in COVERAGE_COLUMNS
)

assert len(statistical_columns) == 27

assert current_team_vectors[TEAM_ID_COLUMN].is_unique

assert not current_team_vectors[
    statistical_columns
].isna().any().any()

print("Current Team Vector structure validated successfully.")

Current Team Vector structure validated successfully.


## Available Teams

In [14]:
available_teams = sorted(
    current_team_vectors[TEAM_ID_COLUMN].unique()
)

print("Available teams:", len(available_teams))
print()
print(available_teams)

Available teams: 48

['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia And Herzegovina', 'Brazil', 'Cabo Verde', 'Canada', 'Colombia', 'Congo DR', 'Croatia', 'Curaçao', 'Czechia', "Côte D'Ivoire", 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'IR Iran', 'Iraq', 'Japan', 'Jordan', 'Korea Republic', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Türkiye', 'USA', 'Uruguay', 'Uzbekistan']


## Retrieve Team Vector

A utility function is created to retrieve the statistical vector of a single national team.

The function:

- Validates the requested country.
- Ensures that exactly one team vector is found.
- Returns only the statistical features.
- Preserves the original feature order.

### Español

**Recuperación del Team Vector** 

Se crea una función auxiliar para recuperar el vector estadístico de una selección.

La función:

- Valida la selección solicitada.
- Garantiza que se encuentre exactamente un vector.
- Devuelve únicamente las variables estadísticas.
- Conserva el orden original de las features.

## Test with one team

In [15]:
argentina_vector = get_team_vector(
    team_name="Argentina",
    team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
)

argentina_vector

50/50                  91.0
Bad Behaviour          28.0
Ball Receipt*       26859.0
Ball Recovery        1799.0
Block                 591.0
Carry               23232.0
Clearance             449.0
Dispossessed          632.0
Dribble              1170.0
Dribbled Past         329.0
Duel                  963.0
Foul Committed        444.0
Foul Won              801.0
Goal Keeper           911.0
Interception          313.0
Miscontrol            507.0
Pass                27016.0
Player Off             18.0
Player On              18.0
Pressure             4893.0
Shield                 19.0
Shot                 1022.0
Error                  11.0
Offside                 5.0
Own Goal Against        2.0
Camera On               0.0
Own Goal For            0.0
Name: 1, dtype: object

In [16]:
assert isinstance(argentina_vector, pd.Series)

assert len(argentina_vector) == len(
    statistical_columns
)

assert argentina_vector.index.tolist() == (
    statistical_columns
)

assert not argentina_vector.isna().any()

print("Team vector retrieved successfully.")

Team vector retrieved successfully.


In [17]:
try:
    get_team_vector(
        team_name="Invalid Team",
        team_vectors=current_team_vectors,
        statistical_columns=statistical_columns,
    )

except ValueError as error:
    print(error)

Team 'Invalid Team' was not found. Available teams: ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia And Herzegovina', 'Brazil', 'Cabo Verde', 'Canada', 'Colombia', 'Congo DR', 'Croatia', 'Curaçao', 'Czechia', "Côte D'Ivoire", 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'IR Iran', 'Iraq', 'Japan', 'Jordan', 'Korea Republic', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Türkiye', 'USA', 'Uruguay', 'Uzbekistan']


# Build Current Match Vector

A match-level representation is constructed by combining the statistical vectors of two current national teams.

The function:

- Retrieves the home and away team vectors.
- Adds `home_` and `away_` prefixes.
- Combines both representations into a single row.
- Preserves the team names as match metadata.
- Produces the base structure required for feature engineering.

### Español

Se construye una representación a nivel partido combinando los vectores estadísticos de dos selecciones actuales.

La función:

- Recupera los vectores del equipo local y visitante.
- Agrega los prefijos `home_` y `away_`.
- Combina ambas representaciones en una única fila.
- Conserva los nombres de las selecciones como metadata del partido.
- Genera la estructura base necesaria para la ingeniería de variables.

## Test with Argentina vs. France

In [18]:
base_match_vector = build_current_match_vector(
    home_team="Argentina",
    away_team="France",
    team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
)

base_match_vector

,home_team,away_team,home_50/50,home_Bad Behaviour,home_Ball Receipt*,home_Ball Recovery,home_Block,home_Carry,home_Clearance,home_Dispossessed,...,away_Player Off,away_Player On,away_Pressure,away_Shield,away_Shot,away_Error,away_Offside,away_Own Goal Against,away_Camera On,away_Own Goal For
0,Argentina,France,91.0,28.0,26859.0,1799.0,591.0,23232.0,449.0,632.0,...,19.0,19.0,3755.0,23.0,531.0,9.0,3.0,0.0,0.0,0.0


In [19]:
expected_home_columns = [
    f"home_{column}"
    for column in statistical_columns
]

expected_away_columns = [
    f"away_{column}"
    for column in statistical_columns
]

assert base_match_vector.shape == (
    1,
    2 + 2 * len(statistical_columns),
)

assert base_match_vector.loc[0, "home_team"] == "Argentina"
assert base_match_vector.loc[0, "away_team"] == "France"

assert all(
    column in base_match_vector.columns
    for column in expected_home_columns
)

assert all(
    column in base_match_vector.columns
    for column in expected_away_columns
)

assert not base_match_vector[
    expected_home_columns + expected_away_columns
].isna().any().any()

print("Base current match vector built successfully.")

Base current match vector built successfully.


In [20]:
try:
    build_current_match_vector(
        home_team="Argentina",
        away_team="Argentina",
        team_vectors=current_team_vectors,
        statistical_columns=statistical_columns,
    )

except ValueError as error:
    print(error)

Home team and away team must be different.


## Prepare Numeric Match Features

### Generate Engineered Match Features

The same feature engineering logic used for the historical match dataset is reused for current matches.

Before generating the features, all home and away statistical columns are explicitly converted to numeric values.

Three feature families are created:

- Absolute differences.
- Relative differences.
- Combined sums.

#### Español

##### Generación de features del partido

Se reutiliza la misma lógica de ingeniería de variables aplicada al dataset histórico de partidos.

Antes de generar las features, todas las columnas estadísticas del equipo local y visitante se convierten explícitamente a valores numéricos.

Se crean tres familias de variables:

- Diferencias absolutas.
- Diferencias relativas.
- Sumas combinadas.

### Convert Statistical Columns

In [25]:
match_statistical_columns = (
    expected_home_columns
    + expected_away_columns
)

base_match_vector[
    match_statistical_columns
] = base_match_vector[
    match_statistical_columns
].astype(float)

In [26]:
assert all(
    pd.api.types.is_numeric_dtype(
        base_match_vector[column]
    )
    for column in match_statistical_columns
)

print("Home and away statistics converted to numeric values.")

Home and away statistics converted to numeric values.


### Detect Valid Statistics

In [27]:
detected_statistics = get_team_statistics(
    base_match_vector
)

print(
    "Detected home-away statistics:",
    len(detected_statistics),
)

print()
print(detected_statistics)

Detected home-away statistics: 27

['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Camera On', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Error', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Offside', 'Own Goal Against', 'Own Goal For', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot']


In [28]:
assert set(detected_statistics) == set(
    statistical_columns
)

print("All statistical features detected successfully.")

All statistical features detected successfully.


In [29]:
current_match_vector = (
    base_match_vector
    .pipe(create_difference_features)
    .pipe(create_relative_difference_features)
    .pipe(create_sum_features)
)

In [30]:
print(
    "Base match vector:",
    base_match_vector.shape,
)

print(
    "Engineered match vector:",
    current_match_vector.shape,
)

Base match vector: (1, 56)
Engineered match vector: (1, 137)


### Integrity Checks

In [31]:
difference_columns = [
    f"diff_{stat}"
    for stat in statistical_columns
]

relative_difference_columns = [
    f"relative_diff_{stat}"
    for stat in statistical_columns
]

sum_columns = [
    f"sum_{stat}"
    for stat in statistical_columns
]

In [32]:
assert current_match_vector.shape == (
    1,
    2 + 5 * len(statistical_columns),
)

assert all(
    column in current_match_vector.columns
    for column in difference_columns
)

assert all(
    column in current_match_vector.columns
    for column in relative_difference_columns
)

assert all(
    column in current_match_vector.columns
    for column in sum_columns
)

engineered_columns = (
    difference_columns
    + relative_difference_columns
    + sum_columns
)

assert not current_match_vector[
    engineered_columns
].isna().any().any()

assert np.isfinite(
    current_match_vector[
        engineered_columns
    ].to_numpy(dtype=float)
).all()

print("Current match features generated successfully.")

Current match features generated successfully.


In [33]:
stat = "Pass"

assert np.isclose(
    current_match_vector.loc[0, f"diff_{stat}"],
    (
        current_match_vector.loc[0, f"home_{stat}"]
        - current_match_vector.loc[0, f"away_{stat}"]
    ),
)

assert np.isclose(
    current_match_vector.loc[0, f"sum_{stat}"],
    (
        current_match_vector.loc[0, f"home_{stat}"]
        + current_match_vector.loc[0, f"away_{stat}"]
    ),
)

print("Feature formulas validated successfully.")

Feature formulas validated successfully.


# Feature Alignment

The current match vector must reproduce the same feature space used during historical model training.

This section:

- Inspects the historical match-vector structure.
- Identifies predictor and metadata columns.
- Compares historical and current feature sets.
- Detects missing and unexpected features.
- Reorders the current vector according to the historical feature schema.
- Produces a prediction-ready match representation.

### Español

El Current Match Vector debe reproducir el mismo espacio de variables utilizado durante el entrenamiento histórico de los modelos.

Esta sección:

- Inspecciona la estructura del Match Vector histórico.
- Identifica las columnas predictoras y de metadata.
- Compara los conjuntos de features históricos y actuales.
- Detecta variables faltantes e inesperadas.
- Reordena el vector actual según el esquema histórico.
- Genera una representación de partido lista para predicción.

## Inspect Historical Match Vector

Before defining the historical feature schema, the dataset columns and data types are inspected.

This prevents metadata, identifiers, scores, or target variables from being accidentally included as model predictors.

In [34]:
print(
    "Historical Match Vector shape:",
    historical_match_vectors.shape,
)

display(
    historical_match_vectors.head()
)

Historical Match Vector shape: (128, 134)


,match_id,match_date,kick_off,year,home_team,away_team,home_score,away_score,competition_stage,home_Ball Receipt*,...,sum_Interception,sum_Miscontrol,sum_Offside,sum_Own Goal Against,sum_Pass,sum_Player Off,sum_Player On,sum_Pressure,sum_Shield,sum_Shot
0,7525,2018-06-14,15:00:00.000,2018,Russia,Saudi Arabia,5,0,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,7578,2018-06-15,12:00:00.000,2018,Egypt,Uruguay,0,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7577,2018-06-15,15:00:00.000,2018,Morocco,Iran,0,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7576,2018-06-15,18:00:00.000,2018,Portugal,Spain,3,3,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,7530,2018-06-16,10:00:00.000,2018,France,Australia,2,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
historical_match_vectors.info()

<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Columns: 134 entries, match_id to sum_Shot
dtypes: float64(124), int64(5), str(5)
memory usage: 134.1 KB


In [36]:
for index, column in enumerate(
    historical_match_vectors.columns
):
    print(
        f"{index:>3} | "
        f"{column:<40} | "
        f"{historical_match_vectors[column].dtype}"
    )

  0 | match_id                                 | int64
  1 | match_date                               | str
  2 | kick_off                                 | str
  3 | year                                     | int64
  4 | home_team                                | str
  5 | away_team                                | str
  6 | home_score                               | int64
  7 | away_score                               | int64
  8 | competition_stage                        | str
  9 | home_Ball Receipt*                       | float64
 10 | home_Ball Recovery                       | float64
 11 | home_Block                               | float64
 12 | home_Carry                               | float64
 13 | home_Clearance                           | float64
 14 | home_Dispossessed                        | float64
 15 | home_Dribble                             | float64
 16 | home_Dribbled Past                       | float64
 17 | home_Duel                                | float64
 1

## Split Columns by Family

In [37]:
historical_column_groups = {
    "home": [
        column
        for column in historical_match_vectors.columns
        if column.startswith("home_")
    ],
    "away": [
        column
        for column in historical_match_vectors.columns
        if column.startswith("away_")
    ],
    "difference": [
        column
        for column in historical_match_vectors.columns
        if column.startswith("diff_")
    ],
    "relative_difference": [
        column
        for column in historical_match_vectors.columns
        if column.startswith("relative_diff_")
    ],
    "sum": [
        column
        for column in historical_match_vectors.columns
        if column.startswith("sum_")
    ],
}

In [38]:
for group_name, columns in historical_column_groups.items():
    print(
        f"{group_name:<22}: {len(columns)}"
    )

home                  : 27
away                  : 27
difference            : 25
relative_difference   : 24
sum                   : 25


In [39]:
grouped_columns = {
    column
    for columns in historical_column_groups.values()
    for column in columns
}

historical_ungrouped_columns = [
    column
    for column in historical_match_vectors.columns
    if column not in grouped_columns
]

print(
    "Ungrouped historical columns:",
    len(historical_ungrouped_columns),
)

print()

for column in historical_ungrouped_columns:
    print(
        f"{column:<40} | "
        f"{historical_match_vectors[column].dtype}"
    )

Ungrouped historical columns: 6

match_id                                 | int64
match_date                               | str
kick_off                                 | str
year                                     | int64
competition_stage                        | str
target                                   | int64


In [40]:
non_numeric_prefixed_columns = [
    column
    for column in grouped_columns
    if not pd.api.types.is_numeric_dtype(
        historical_match_vectors[column]
    )
]

print(
    "Non-numeric prefixed columns:",
    len(non_numeric_prefixed_columns),
)

print()

print(
    sorted(non_numeric_prefixed_columns)
)

Non-numeric prefixed columns: 2

['away_team', 'home_team']


In [41]:
potential_metadata_keywords = {
    "team",
    "score",
    "result",
    "target",
    "match",
    "date",
    "year",
    "stage",
}

potential_prefixed_metadata = [
    column
    for column in grouped_columns
    if any(
        keyword in column.lower()
        for keyword in potential_metadata_keywords
    )
]

print(
    "Potential prefixed metadata columns:",
    len(potential_prefixed_metadata),
)

print()

print(
    sorted(potential_prefixed_metadata)
)

Potential prefixed metadata columns: 4

['away_score', 'away_team', 'home_score', 'home_team']


## Current Match Vector Inspection

In [42]:
print(
    "Current Match Vector shape:",
    current_match_vector.shape,
)

print()

for index, column in enumerate(
    current_match_vector.columns
):
    print(
        f"{index:>3} | "
        f"{column:<40} | "
        f"{current_match_vector[column].dtype}"
    )

Current Match Vector shape: (1, 137)

  0 | home_team                                | object
  1 | away_team                                | object
  2 | home_50/50                               | float64
  3 | home_Bad Behaviour                       | float64
  4 | home_Ball Receipt*                       | float64
  5 | home_Ball Recovery                       | float64
  6 | home_Block                               | float64
  7 | home_Carry                               | float64
  8 | home_Clearance                           | float64
  9 | home_Dispossessed                        | float64
 10 | home_Dribble                             | float64
 11 | home_Dribbled Past                       | float64
 12 | home_Duel                                | float64
 13 | home_Foul Committed                      | float64
 14 | home_Foul Won                            | float64
 15 | home_Goal Keeper                         | float64
 16 | home_Interception                        | flo

In [43]:
current_column_groups = {
    "home": [
        column
        for column in current_match_vector.columns
        if column.startswith("home_")
    ],
    "away": [
        column
        for column in current_match_vector.columns
        if column.startswith("away_")
    ],
    "difference": [
        column
        for column in current_match_vector.columns
        if column.startswith("diff_")
    ],
    "relative_difference": [
        column
        for column in current_match_vector.columns
        if column.startswith("relative_diff_")
    ],
    "sum": [
        column
        for column in current_match_vector.columns
        if column.startswith("sum_")
    ],
}

for group_name, columns in current_column_groups.items():
    print(
        f"{group_name:<22}: {len(columns)}"
    )

home                  : 28
away                  : 28
difference            : 27
relative_difference   : 27
sum                   : 27


## Build Historical Feature Schema

The historical feature schema is reconstructed from the dataset used during model development.

Metadata, identifiers, scores, and the target variable are excluded.

The remaining columns define:

- The historical predictor set.
- The expected feature names.
- The exact feature order required for model inference.

#### Español

Se reconstruye el esquema histórico de variables a partir del dataset utilizado durante el desarrollo de los modelos.

Se excluyen la metadata, los identificadores, los marcadores y la variable target.

Las columnas restantes definen:

- El conjunto histórico de predictores.
- Los nombres de features esperados.
- El orden exacto requerido para la inferencia del modelo.

### Define Historical Metadata

In [45]:
historical_metadata_columns = [
    "match_id",
    "match_date",
    "kick_off",
    "year",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "competition_stage",
    "target",
]

In [46]:
missing_metadata_columns = [
    column
    for column in historical_metadata_columns
    if column not in historical_match_vectors.columns
]

assert not missing_metadata_columns, (
    "Missing historical metadata columns: "
    f"{missing_metadata_columns}"
)

print("Historical metadata columns validated.")

Historical metadata columns validated.


### Extract Historical Features

In [47]:
historical_feature_columns = [
    column
    for column in historical_match_vectors.columns
    if column not in historical_metadata_columns
]

In [48]:
print(
    "Historical feature count:",
    len(historical_feature_columns),
)

print()

print(historical_feature_columns)

Historical feature count: 124

['home_Ball Receipt*', 'home_Ball Recovery', 'home_Block', 'home_Carry', 'home_Clearance', 'home_Dispossessed', 'home_Dribble', 'home_Dribbled Past', 'home_Duel', 'home_Error', 'home_Foul Committed', 'home_Foul Won', 'home_Goal Keeper', 'home_Interception', 'home_Miscontrol', 'home_Own Goal Against', 'home_Pass', 'home_Player Off', 'home_Player On', 'home_Pressure', 'home_Shot', 'home_Offside', 'home_Shield', 'home_Bad Behaviour', 'home_50/50', 'away_Ball Receipt*', 'away_Ball Recovery', 'away_Block', 'away_Carry', 'away_Clearance', 'away_Dispossessed', 'away_Dribble', 'away_Dribbled Past', 'away_Duel', 'away_Error', 'away_Foul Committed', 'away_Foul Won', 'away_Goal Keeper', 'away_Interception', 'away_Miscontrol', 'away_Own Goal Against', 'away_Pass', 'away_Player Off', 'away_Player On', 'away_Pressure', 'away_Shot', 'away_Offside', 'away_Shield', 'away_Bad Behaviour', 'away_50/50', 'diff_50/50', 'diff_Bad Behaviour', 'diff_Ball Receipt*', 'diff_Ball Rec

### Validate Historical Features

In [49]:
assert historical_feature_columns

assert len(historical_feature_columns) == len(
    set(historical_feature_columns)
)

assert all(
    pd.api.types.is_numeric_dtype(
        historical_match_vectors[column]
    )
    for column in historical_feature_columns
)

assert not historical_match_vectors[
    historical_feature_columns
].isna().any().any()

print(
    "Historical feature schema built successfully."
)

Historical feature schema built successfully.


## Build Current Feature Set

Current match metadata is excluded so that only model-ready statistical features remain.

The resulting feature set is then compared against the historical schema.

In [50]:
current_metadata_columns = [
    "home_team",
    "away_team",
]

In [51]:
current_feature_columns = [
    column
    for column in current_match_vector.columns
    if column not in current_metadata_columns
]

In [52]:
print(
    "Current feature count:",
    len(current_feature_columns),
)

Current feature count: 135


In [54]:
assert len(current_feature_columns) == len(
    set(current_feature_columns)
)

assert all(
    pd.api.types.is_numeric_dtype(
        current_match_vector[column]
    )
    for column in current_feature_columns
)

assert not current_match_vector[
    current_feature_columns
].isna().any().any()

print(
    "Current feature set built successfully."
)

Current feature set built successfully.


## Compare Historical and Current Features

The historical and current feature sets are compared by name.

This comparison identifies:

- Missing features required by the historical model.
- Extra features generated by the current pipeline.
- Features shared by both datasets.

In [55]:
historical_feature_set = set(
    historical_feature_columns
)

current_feature_set = set(
    current_feature_columns
)

In [56]:
missing_features = sorted(
    historical_feature_set
    - current_feature_set
)

extra_features = sorted(
    current_feature_set
    - historical_feature_set
)

common_features = sorted(
    historical_feature_set
    & current_feature_set
)

In [57]:
print(
    "Historical features:",
    len(historical_feature_set),
)

print(
    "Current features:",
    len(current_feature_set),
)

print(
    "Common features:",
    len(common_features),
)

print(
    "Missing features:",
    len(missing_features),
)

print(
    "Extra features:",
    len(extra_features),
)

Historical features: 124
Current features: 135
Common features: 124
Missing features: 0
Extra features: 11


In [58]:
print("Missing features:")
print(missing_features)

print()

print("Extra features:")
print(extra_features)

Missing features:
[]

Extra features:
['away_Camera On', 'away_Own Goal For', 'diff_Camera On', 'diff_Own Goal For', 'home_Camera On', 'home_Own Goal For', 'relative_diff_Ball Receipt*', 'relative_diff_Camera On', 'relative_diff_Own Goal For', 'sum_Camera On', 'sum_Own Goal For']


## Prediction Ready Match Vector

In [59]:
prediction_ready_match_vector = (
    current_match_vector[
        current_metadata_columns
        + historical_feature_columns
    ]
    .copy()
)

In [65]:
assert list(
    prediction_ready_match_vector.columns[2:]
) == historical_feature_columns

assert prediction_ready_match_vector.shape == (
    1,
    2 + len(historical_feature_columns),
)

assert not prediction_ready_match_vector[
    historical_feature_columns
].isna().any().any()

assert prediction_ready_match_vector.loc[
    0,
    "home_team",
] == "Argentina"

assert prediction_ready_match_vector.loc[
    0,
    "away_team",
] == "France"

print(
    "Prediction-ready match vector validated successfully."
)

Prediction-ready match vector validated successfully.


# Export Prediction-Ready Match Vector

The prediction-ready match vector is exported for downstream inference.

The exported dataset:

- Preserves the historical feature order.
- Contains only the features required by the historical model.
- Is ready to be consumed by the prediction pipeline.

### Español

El Prediction-Ready Match Vector se exporta para ser utilizado posteriormente durante la inferencia.

El dataset exportado:

- Conserva el orden histórico de las features.
- Contiene únicamente las variables requeridas por el modelo histórico.
- Queda listo para ser utilizado por el pipeline de predicción.

## Export Code

In [66]:
prediction_ready_match_vector.to_csv(
    OUTPUT_PATH,
    index=False,
)

In [67]:
assert OUTPUT_PATH.exists()

reloaded_match_vector = pd.read_csv(
    OUTPUT_PATH
)

In [68]:
assert (
    reloaded_match_vector.shape
    == prediction_ready_match_vector.shape
)

assert list(
    reloaded_match_vector.columns
) == list(
    prediction_ready_match_vector.columns
)

assert not reloaded_match_vector.isna().any().any()

print("Export checks passed.")

Export checks passed.


In [69]:
print(
    "Prediction-ready match vector:",
    prediction_ready_match_vector.shape,
)

print(
    "Reloaded match vector:",
    reloaded_match_vector.shape,
)

Prediction-ready match vector: (1, 126)
Reloaded match vector: (1, 126)


# Test Master Function

In [71]:
test_vector = build_prediction_ready_match_vector(
    home_team="Argentina",
    away_team="France",
    team_vectors=current_team_vectors,
    statistical_columns=statistical_columns,
    historical_feature_columns=historical_feature_columns,
)

In [72]:
assert test_vector.equals(
    prediction_ready_match_vector
)

# Discussion

### English

- This notebook bridges the gap between historical model development and future match inference.

- A complete inference pipeline was implemented to transform current team vectors into prediction-ready match vectors using the same feature engineering strategy applied during model training.

- Feature alignment revealed that every historical predictor was successfully reproduced by the current pipeline. Eleven additional features were generated due to the expanded player database, primarily related to newly available event types. Since these variables were not present during historical training, they were excluded from the prediction-ready representation while preserving the original feature schema.

- The final orchestration function encapsulates the complete transformation process, allowing future notebooks to generate prediction-ready vectors from any pair of national teams through a single function call. This significantly improves code reusability, reproducibility, and maintainability.

### Español

- Esta notebook establece el puente entre el desarrollo histórico de los modelos y la futura etapa de inferencia.

- Se implementó un pipeline completo capaz de transformar los Team Vectors actuales en Match Vectors listos para predicción utilizando exactamente la misma estrategia de ingeniería de variables empleada durante el entrenamiento de los modelos.

- El alineamiento de features demostró que todas las variables históricas fueron reproducidas correctamente por el pipeline actual. Además, se identificaron once variables adicionales generadas como consecuencia de la expansión de la base de jugadores, principalmente asociadas a nuevos tipos de eventos disponibles. Dado que estas variables no participaron del entrenamiento histórico, fueron excluidas del vector final de predicción para preservar el esquema original de features.

- Finalmente, la incorporación de una función de orquestación encapsula todo el proceso de transformación, permitiendo generar vectores listos para inferencia para cualquier enfrentamiento entre selecciones mediante una única llamada a función. Esto mejora considerablemente la reutilización del código, la reproducibilidad y el mantenimiento del proyecto.

# Conclusion

### English

- A complete current match vector generation pipeline was implemented.

- Historical feature engineering was successfully reproduced for current team vectors.

- Historical and current feature schemas were compared and aligned.

- No historical predictors were missing from the current pipeline.

- Additional features generated by the expanded database were correctly identified and excluded from the prediction-ready representation.

- A reusable master function was created to automate the complete inference preparation process.

- The project is now prepared to integrate the trained prediction models in the next development stage.


### Español

- Se implementó un pipeline completo para generar Match Vectors actuales.

- Se reprodujo exitosamente la ingeniería de variables utilizada durante el entrenamiento histórico.

- Se compararon y alinearon los esquemas de features históricos y actuales.

- No se detectaron variables históricas faltantes en el pipeline actual.

- Las nuevas variables generadas por la expansión de la base de datos fueron identificadas y excluidas correctamente del vector final de predicción.

- Se desarrolló una función principal reutilizable que automatiza todo el proceso de preparación para la inferencia.

- El proyecto quedó preparado para integrar los modelos entrenados en la siguiente etapa de desarrollo.
